In [1]:
import importlib
from pathlib import Path
import sys

# Mount Google Drive if running in Google Colab
try:
    from google.colab import drive
    drive.mount("/content/drive")
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

REPO_URL = "https://github.com/pratik0620/adaptive-federated-aggregation.git"

if IN_COLAB or Path("/content").exists():
    REPO_ROOT = Path("/content/adaptive-federated-healthcare")
    # If alternate clone directory exists and primary does not, use it
    if not REPO_ROOT.exists():
        alt_root = Path("/content/adaptive-federated-aggregation")
        if alt_root.exists() and (alt_root / "src").is_dir():
            REPO_ROOT = alt_root
        else:
            !git clone {REPO_URL} {REPO_ROOT}
    
    # If repository already exists, safely sync with remote and discard local dirty diffs
    if REPO_ROOT.exists() and (REPO_ROOT / ".git").is_dir():
        !git -C {REPO_ROOT} fetch origin main
        !git -C {REPO_ROOT} reset --hard origin/main
else:
    REPO_ROOT = Path.cwd()
    if not (REPO_ROOT / "src").is_dir() and (REPO_ROOT.parent / "src").is_dir():
        REPO_ROOT = REPO_ROOT.parent

# Ensure REPO_ROOT takes top precedence in sys.path
repo_path_str = str(REPO_ROOT)
if repo_path_str in sys.path:
    sys.path.remove(repo_path_str)
sys.path.insert(0, repo_path_str)

# Invalidate import caches and clear cached src modules to prevent stale imports
importlib.invalidate_caches()
for module_name in list(sys.modules):
    if module_name == "src" or module_name.startswith("src."):
        del sys.modules[module_name]

SRC_DIR = REPO_ROOT / "src"
if not SRC_DIR.is_dir():
    raise FileNotFoundError(f"src directory not found at {SRC_DIR}")

print("Repository:", REPO_ROOT)
print("Source modules:", SRC_DIR)

Repository: E:\College\Third Year\Data Science\Course Project\Project Repo
Source modules: E:\College\Third Year\Data Science\Course Project\Project Repo\src


In [2]:
# Install the packages required by the notebook and shared preprocessing modules.
%pip install -q numpy pandas scikit-learn matplotlib jupyter


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.0.1 -> 26.2.1
[notice] To update, run: C:\Users\Admin\AppData\Local\Programs\Python\Python310\python.exe -m pip install --upgrade pip


# CheXpert preprocessing

This notebook prepares CheXpert as a separate metadata-only federated client. For the configurable target disease (`Pleural Effusion`), `1` is positive and `0` is negative; `-1` uncertain and blank/not mentioned are excluded (they are never converted to zero, and `No Finding` is not used to infer negatives).

**Split Strategy:**
- Stanford's official CheXpert validation set (`valid.csv`, 234 images) is evaluated by board-certified radiologists as the consensus ground-truth benchmark. Consistent with the NIH preprocessing pipeline that preserves the official NIH test split, the official CheXpert validation set is preserved as the held-out `test` split.
- The clean subset of `train.csv` (121,583 images across 40,809 unique patients) is split at the patient level into `train` (90%) and `val` (10%).
- Strict patient-level and study-level leakage checks ensure zero overlap across all three splits.

In [3]:
import json, platform
from pathlib import Path
import numpy as np
import pandas as pd

from src.config import TARGET_DISEASE, RANDOM_SEED, MAX_POSITIVE_SAMPLES, MAX_NEGATIVE_SAMPLES
from src.dataset_utils import chexpert_clean_binary_labels, chexpert_patient_id, standardize_chexpert
from src.preprocessing_utils import (
    assert_no_leakage,
    assert_no_study_leakage,
    build_summary,
    require_file,
    sample_clean_cases,
    save_splits,
    save_summary,
    set_reproducibility,
    split_train_val_by_patient,
)
set_reproducibility(RANDOM_SEED)
print('Python:', platform.python_version(), 'pandas:', pd.__version__)

# Dataset-specific Google Drive configuration.
BASE_DRIVE_PATH = Path("/content/drive/MyDrive/Healthcare_FL")
if not BASE_DRIVE_PATH.exists() and (REPO_ROOT / "data").exists():
    BASE_DRIVE_PATH = REPO_ROOT

CHEXPERT_ROOT = BASE_DRIVE_PATH / "data" / "raw" / "CheXpert" if (BASE_DRIVE_PATH / "data" / "raw" / "CheXpert").exists() else BASE_DRIVE_PATH / "raw" / "CheXpert"
TRAIN_CSV = CHEXPERT_ROOT / "train.csv"
VALID_CSV = CHEXPERT_ROOT / "valid.csv"
IMAGE_ROOT = CHEXPERT_ROOT
OUTPUT_DIR = BASE_DRIVE_PATH / "data" / "processed" / "CheXpert" if (BASE_DRIVE_PATH / "data").exists() else BASE_DRIVE_PATH / "processed" / "CheXpert"

train_csv_path = require_file(TRAIN_CSV, "CheXpert train metadata CSV")
valid_csv_path = require_file(VALID_CSV, "CheXpert validation metadata CSV")

print("Repository:", REPO_ROOT)
print("Google Drive:", BASE_DRIVE_PATH)
print("Dataset:", CHEXPERT_ROOT)
print("Train Metadata:", TRAIN_CSV)
print("Validation Metadata:", VALID_CSV)
print("Output Directory:", OUTPUT_DIR)
print("Required CheXpert metadata files exist.")

Python: 3.10.11 pandas: 2.3.3
Repository: E:\College\Third Year\Data Science\Course Project\Project Repo
Google Drive: E:\College\Third Year\Data Science\Course Project\Project Repo
Dataset: E:\College\Third Year\Data Science\Course Project\Project Repo\data\raw\CheXpert
Train Metadata: E:\College\Third Year\Data Science\Course Project\Project Repo\data\raw\CheXpert\train.csv
Validation Metadata: E:\College\Third Year\Data Science\Course Project\Project Repo\data\raw\CheXpert\valid.csv
Output Directory: E:\College\Third Year\Data Science\Course Project\Project Repo\data\processed\CheXpert
Required CheXpert metadata files exist.


## Load and inspect

CheXpert contains separate label columns per pathology. We inspect both `train.csv` and `valid.csv`, and filter them strictly using the label policy:
- `Pleural Effusion == 1.0` -> `target = 1`
- `Pleural Effusion == 0.0` -> `target = 0`
- `Pleural Effusion == -1.0` -> excluded (uncertain)
- `Pleural Effusion == NaN` -> excluded (unmentioned/missing)

In [4]:
chexpert_train = pd.read_csv(train_csv_path)
chexpert_valid = pd.read_csv(valid_csv_path)

target_column = TARGET_DISEASE
if target_column not in chexpert_train.columns:
    raise KeyError(f'CheXpert target column {target_column!r} is absent from train. Available: {chexpert_train.columns.tolist()}')
if target_column not in chexpert_valid.columns:
    raise KeyError(f'CheXpert target column {target_column!r} is absent from valid. Available: {chexpert_valid.columns.tolist()}')

print('train.csv rows (studies/images):', len(chexpert_train))
print('valid.csv rows (studies/images):', len(chexpert_valid))
print('total raw images:', len(chexpert_train) + len(chexpert_valid))
print('columns:', chexpert_train.columns.tolist())

display(chexpert_train.isna().sum().sort_values(ascending=False).head(20).to_frame('missing_train'))
display(chexpert_train[target_column].value_counts(dropna=False).sort_index().to_frame('raw_train_target_distribution'))
display(chexpert_valid[target_column].value_counts(dropna=False).sort_index().to_frame('raw_valid_target_distribution'))

clean_train, train_counts = chexpert_clean_binary_labels(chexpert_train, target_column)
clean_valid, valid_counts = chexpert_clean_binary_labels(chexpert_valid, target_column)

print('Train counts before filtering:', train_counts)
print('Valid counts before filtering:', valid_counts)
print('Clean train rows:', len(clean_train))
print('Clean valid rows:', len(clean_valid))
print('Clean total rows:', len(clean_train) + len(clean_valid))

train.csv rows (studies/images): 223414
valid.csv rows (studies/images): 234
total raw images: 223648
columns: ['Path', 'Sex', 'Age', 'Frontal/Lateral', 'AP/PA', 'No Finding', 'Enlarged Cardiomediastinum', 'Cardiomegaly', 'Lung Opacity', 'Lung Lesion', 'Edema', 'Consolidation', 'Pneumonia', 'Atelectasis', 'Pneumothorax', 'Pleural Effusion', 'Pleural Other', 'Fracture', 'Support Devices']


,missing_train
Pleural Other,216922
Lung Lesion,211470
Fracture,211220
No Finding,201033
Pneumonia,195806
Enlarged Cardiomediastinum,178575
Cardiomegaly,177211
Atelectasis,154971
Consolidation,152792
Pneumothorax,144480


,raw_train_target_distribution
Pleural Effusion,
-1.0,11628
0.0,35396
1.0,86187
NaN,90203


,raw_valid_target_distribution
Pleural Effusion,
0.0,167
1.0,67


Train counts before filtering: {'positive_before_filter': 86187, 'negative_before_filter': 35396, 'uncertain_before_filter': 11628, 'missing_not_mentioned_before_filter': 90203}
Valid counts before filtering: {'positive_before_filter': 67, 'negative_before_filter': 167, 'uncertain_before_filter': 0, 'missing_not_mentioned_before_filter': 0}
Clean train rows: 121583
Clean valid rows: 234
Clean total rows: 121817


In [5]:
# Standardize both train and valid metadata into the standard client schema.
# Paths are kept reproducible and relative with forward slashes; machine-specific absolute prefixes are avoided.
standardized_train = standardize_chexpert(clean_train)
standardized_valid = standardize_chexpert(clean_valid)

# Verify zero patient and study overlap between raw/clean train and valid
raw_train_pts = set(chexpert_patient_id(chexpert_train['Path']))
raw_valid_pts = set(chexpert_patient_id(chexpert_valid['Path']))
assert raw_train_pts.isdisjoint(raw_valid_pts), f'Patient overlap between raw train and valid: {len(raw_train_pts & raw_valid_pts)}'

# Optional sampling applied to the training partition
standardized_train = sample_clean_cases(
    standardized_train,
    max_positive=MAX_POSITIVE_SAMPLES,
    max_negative=MAX_NEGATIVE_SAMPLES,
    seed=RANDOM_SEED,
)
print('Clean train rows after optional sampling:', len(standardized_train))
print('Unique patients in train+val pool:', standardized_train['patient_id'].nunique())
display(standardized_train.head())

# Split the clean training set at patient level into train (90%) and val (10%).
# The official CheXpert validation set (radiologist consensus) is preserved as the held-out test split.
train_val_splits = split_train_val_by_patient(standardized_train, seed=RANDOM_SEED, val_size=0.1)

splits = {
    'train': train_val_splits['train'],
    'val': train_val_splits['val'],
    'test': standardized_valid,
}

# Strict leakage checks across all splits
assert_no_leakage(splits)
assert_no_study_leakage(splits)
print('patient and study leakage checks: passed')

for split_name, split_frame in splits.items():
    print(
        f'{split_name}: {len(split_frame)} images | '
        f'{split_frame["patient_id"].nunique()} patients | '
        f'{split_frame["study_id"].nunique()} studies | '
        f'positive: {int((split_frame["target"] == 1).sum())} | '
        f'negative: {int((split_frame["target"] == 0).sum())}'
    )

display(pd.DataFrame([
    {
        'split': split_name,
        'images': len(split_frame),
        'patients': split_frame['patient_id'].nunique(),
        'studies': split_frame['study_id'].nunique(),
        'positive': int((split_frame['target'] == 1).sum()),
        'negative': int((split_frame['target'] == 0).sum()),
    }
    for split_name, split_frame in splits.items()
]))

save_splits(splits, OUTPUT_DIR)

summary = build_summary(
    splits,
    'CheXpert',
    {
        'target_disease': TARGET_DISEASE,
        'source_target_label': TARGET_DISEASE,
        'random_seed': RANDOM_SEED,
        'max_positive_samples': MAX_POSITIVE_SAMPLES,
        'max_negative_samples': MAX_NEGATIVE_SAMPLES,
        'train_filter_counts_before_filtering': train_counts,
        'valid_filter_counts_before_filtering': valid_counts,
        'official_validation_preserved_as_test': True,
        'split_strategy': 'official_validation_as_test_train_split_90_10_patient_level',
        'image_paths_materialized': False,
    },
)
save_summary(summary, OUTPUT_DIR)
print(json.dumps(summary, indent=2))
print('Saved files:', sorted([path.name for path in OUTPUT_DIR.glob('*')]))

Clean train rows after optional sampling: 121583
Unique patients in train+val pool: 40809


,image_id,patient_id,study_id,source,target,age,sex,view_position,projection,image_width,image_height,pixel_spacing_x,pixel_spacing_y,scanner_manufacturer,image_path,finding_labels,follow_up
0,CheXpert-v1.0-small/train/patient10592/study1/...,patient10592,patient10592_study1,CheXpert,1,73,Male,Frontal,PA,NaN,NaN,NaN,NaN,NaN,CheXpert-v1.0-small/train/patient10592/study1/...,NaN,NaN
1,CheXpert-v1.0-small/train/patient50515/study2/...,patient50515,patient50515_study2,CheXpert,1,88,Male,Frontal,AP,NaN,NaN,NaN,NaN,NaN,CheXpert-v1.0-small/train/patient50515/study2/...,NaN,NaN
2,CheXpert-v1.0-small/train/patient23056/study1/...,patient23056,patient23056_study1,CheXpert,1,86,Female,Frontal,AP,NaN,NaN,NaN,NaN,NaN,CheXpert-v1.0-small/train/patient23056/study1/...,NaN,NaN
3,CheXpert-v1.0-small/train/patient22899/study1/...,patient22899,patient22899_study1,CheXpert,0,55,Male,Lateral,NaN,NaN,NaN,NaN,NaN,NaN,CheXpert-v1.0-small/train/patient22899/study1/...,NaN,NaN
4,CheXpert-v1.0-small/train/patient43896/study2/...,patient43896,patient43896_study2,CheXpert,1,81,Female,Frontal,AP,NaN,NaN,NaN,NaN,NaN,CheXpert-v1.0-small/train/patient43896/study2/...,NaN,NaN


patient and study leakage checks: passed
train: 109572 images | 36728 patients | 90675 studies | positive: 77719 | negative: 31853
val: 12011 images | 4081 patients | 9853 studies | positive: 8468 | negative: 3543
test: 234 images | 200 patients | 200 studies | positive: 67 | negative: 167


,split,images,patients,studies,positive,negative
0,train,109572,36728,90675,77719,31853
1,val,12011,4081,9853,8468,3543
2,test,234,200,200,67,167


{
  "source": "CheXpert",
  "parameters": {
    "target_disease": "Pleural Effusion",
    "source_target_label": "Pleural Effusion",
    "random_seed": 42,
    "max_positive_samples": null,
    "max_negative_samples": null,
    "train_filter_counts_before_filtering": {
      "positive_before_filter": 86187,
      "negative_before_filter": 35396,
      "uncertain_before_filter": 11628,
      "missing_not_mentioned_before_filter": 90203
    },
    "valid_filter_counts_before_filtering": {
      "positive_before_filter": 67,
      "negative_before_filter": 167,
      "uncertain_before_filter": 0,
      "missing_not_mentioned_before_filter": 0
    },
    "official_validation_preserved_as_test": true,
    "split_strategy": "official_validation_as_test_train_split_90_10_patient_level",
    "image_paths_materialized": false
  },
  "splits": {
    "train": {
      "images": 109572,
      "patients": 36728,
      "studies": 90675,
      "positive": 77719,
      "negative": 31853
    },
    "val